# Backbone: SeaDroneSee RGB
YOLOv8n trained on SeaDroneSee, class swimmer->0 (person), nc=1

In [ ]:
import subprocess, sys
for pkg in ['ultralytics', 'optuna', 'pandas', 'matplotlib']:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print('Packages OK')

In [ ]:
import os, sys, gc, json, glob, shutil, random, math, time
import xml.etree.ElementTree as ET
from collections import defaultdict
import torch
import optuna
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from ultralytics import YOLO

optuna.logging.set_verbosity(optuna.logging.WARNING)
BASE_DIR     = '/root/AIP491'
BACKBONE_DIR = os.path.join(BASE_DIR, 'backbones')
DEVICE       = 0
NUM_WORKERS  = 4
IMG_SIZE     = 640
os.makedirs(BACKBONE_DIR, exist_ok=True)
print(f'torch {torch.__version__} | CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
SDS_DIR       = os.path.join(BASE_DIR, 'data', 'seadronessee', 'compressed')
DATA_DIR      = os.path.join(BASE_DIR, 'data', 'sds_rgb_yolo')
RUNS_DIR      = os.path.join(BACKBONE_DIR, 'runs', 'sds_rgb')
BACKBONE_PATH = os.path.join(BACKBONE_DIR, 'SDS_RGB_best.pt')
YAML_PATH     = os.path.join(DATA_DIR, 'dataset.yaml')
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RUNS_DIR, exist_ok=True)

EPOCHS        = 50
PATIENCE      = 5
BATCH         = 16
LR0           = 5e-3
LRF           = 0.01
MOMENTUM      = 0.937
WEIGHT_DECAY  = 5e-4
WARMUP_EPOCHS = 3

# SeaDroneSee: giu lai 'swimmer' -> class 0 (person in water)
KEEP_CLASS = 'swimmer'
print(f'Backbone: SeaDroneSee RGB | keep_class: {KEEP_CLASS} -> 0')

In [ ]:
# === Chuyen SeaDroneSee COCO JSON -> YOLO format ===
def coco_to_yolo_single_class(coco_json_path, images_dir, out_labels_dir, keep_name='swimmer'):
    os.makedirs(out_labels_dir, exist_ok=True)
    with open(coco_json_path) as f:
        coco = json.load(f)

    # Tim category id cua keep_name
    keep_id = None
    for c in coco.get('categories', []):
        if c.get('name') == keep_name:
            keep_id = c['id']
            break
    if keep_id is None:
        raise ValueError(f'Category "{keep_name}" not found. Available: {[c["name"] for c in coco["categories"]]}')

    # Gom annotation theo image_id
    anns_by_img = defaultdict(list)
    for ann in coco.get('annotations', []):
        if ann.get('iscrowd', 0) == 1:
            continue
        if ann.get('category_id') != keep_id:
            continue
        bbox = ann.get('bbox')
        if not bbox or bbox[2] <= 0 or bbox[3] <= 0:
            continue
        anns_by_img[ann['image_id']].append(ann)

    img_map = {img['id']: img for img in coco.get('images', [])}
    written, skipped = 0, 0

    for img_id, img_info in img_map.items():
        w, h = img_info.get('width', 0), img_info.get('height', 0)
        if not w or not h:
            continue
        fname = os.path.basename(img_info['file_name'])
        img_path = os.path.join(images_dir, fname)
        if not os.path.exists(img_path):
            skipped += 1
            continue

        anns = anns_by_img.get(img_id, [])
        if not anns:
            continue

        lines = []
        for ann in anns:
            x, y, bw, bh = ann['bbox']
            cx = min(max((x + bw / 2) / w, 0.0), 1.0)
            cy = min(max((y + bh / 2) / h, 0.0), 1.0)
            bw_n = min(max(bw / w, 0.0), 1.0)
            bh_n = min(max(bh / h, 0.0), 1.0)
            if bw_n <= 0 or bh_n <= 0:
                continue
            lines.append(f'0 {cx:.6f} {cy:.6f} {bw_n:.6f} {bh_n:.6f}')

        if not lines:
            continue
        stem = os.path.splitext(fname)[0]
        with open(os.path.join(out_labels_dir, stem + '.txt'), 'w') as f:
            f.write('\n'.join(lines))
        written += 1

    print(f'  {out_labels_dir}: {written} labels written, {skipped} images not found')
    return written


def prepare_sds_yolo(sds_dir, out_dir, keep_name='swimmer'):
    check = os.path.join(out_dir, 'labels', 'train')
    if os.path.isdir(check) and len(os.listdir(check)) > 0:
        print(f'Dataset da ton tai: {out_dir} ({len(os.listdir(check))} train labels)')
        return

    print(f'Dang chuyen SeaDroneSee -> YOLO (keep: {keep_name})...')
    ann_dir = os.path.join(sds_dir, 'annotations')

    for split in ['train', 'val']:
        coco_json = os.path.join(ann_dir, f'instances_{split}.json')
        img_src   = os.path.join(sds_dir, 'images', split)
        lbl_dst   = os.path.join(out_dir, 'labels', split)
        img_dst   = os.path.join(out_dir, 'images', split)
        os.makedirs(img_dst, exist_ok=True)

        n = coco_to_yolo_single_class(coco_json, img_src, lbl_dst, keep_name=keep_name)

        # Symlink or copy images (chi copy anh co label)
        lbl_stems = {os.path.splitext(f)[0] for f in os.listdir(lbl_dst) if f.endswith('.txt')}
        copied = 0
        for fname in os.listdir(img_src):
            stem = os.path.splitext(fname)[0]
            if stem in lbl_stems:
                src = os.path.join(img_src, fname)
                dst = os.path.join(img_dst, fname)
                if not os.path.exists(dst):
                    shutil.copy2(src, dst)
                copied += 1
        print(f'  {split}: {copied} images copied')


prepare_sds_yolo(SDS_DIR, DATA_DIR, keep_name=KEEP_CLASS)

with open(YAML_PATH, 'w') as f:
    f.write(f'path: {DATA_DIR}\ntrain: images/train\nval: images/val\nnc: 1\nnames: [person]\n')
print(f'YAML: {YAML_PATH}')

In [ ]:
# === Optuna Bayesian Tuning (15 trials) ===
_tune_out = os.path.join(RUNS_DIR, 'tune_best_params.json')

if os.path.exists(_tune_out):
    with open(_tune_out) as _f:
        _best = json.load(_f)
    print('Tune results loaded:', _tune_out)
else:
    def _tune_backbone(trial):
        _lr0 = trial.suggest_float('lr0', 1e-4, 1e-2, log=True)
        _lrf = trial.suggest_float('lrf', 0.01, 0.3)
        _mom = trial.suggest_float('momentum', 0.80, 0.98)
        _wd  = trial.suggest_float('weight_decay', 1e-5, 1e-3, log=True)
        _wu  = trial.suggest_int('warmup_epochs', 1, 5)
        _bs  = trial.suggest_categorical('batch_size', [8, 16])

        try:
            _m = YOLO('yolov8n.pt')
            _r = _m.train(
                data=YAML_PATH, epochs=10, imgsz=IMG_SIZE, batch=_bs,
                lr0=_lr0, lrf=_lrf, momentum=_mom, weight_decay=_wd,
                warmup_epochs=_wu, device=DEVICE, workers=NUM_WORKERS,
                patience=3, exist_ok=True, verbose=False,
                project=os.path.join(RUNS_DIR, 'tune'), name='trial',
                fliplr=0.5, mosaic=1.0, scale=0.5, single_cls=True,
            )
            _map = _r.results_dict.get('metrics/mAP50-95(B)', 0.0)
            del _m; torch.cuda.empty_cache(); gc.collect()
            return _map
        except RuntimeError as e:
            if 'out of memory' in str(e).lower():
                torch.cuda.empty_cache(); gc.collect()
                return 0.0
            raise

    _study = optuna.create_study(
        direction='maximize',
        pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=3),
        study_name='sds_rgb_backbone_tune'
    )
    _study.optimize(_tune_backbone, n_trials=15, show_progress_bar=True)

    _best = dict(_study.best_params)
    _best['best_map'] = _study.best_value
    with open(_tune_out, 'w') as _f:
        json.dump(_best, _f, indent=2)

LR0          = _best.get('lr0', LR0)
LRF          = _best.get('lrf', LRF)
MOMENTUM     = _best.get('momentum', MOMENTUM)
WEIGHT_DECAY = _best.get('weight_decay', WEIGHT_DECAY)
WARMUP_EPOCHS = _best.get('warmup_epochs', WARMUP_EPOCHS)
BATCH        = _best.get('batch_size', BATCH)
print('BEST PARAMS:')
for _k, _v in _best.items():
    print(f'  {_k}: {_v}')

In [ ]:
# === Train final model ===
if os.path.exists(BACKBONE_PATH):
    print(f'Backbone da ton tai: {BACKBONE_PATH}')
else:
    print('Training...')
    model = YOLO('yolov8n.pt')
    results = model.train(
        data=YAML_PATH,
        epochs=EPOCHS, imgsz=IMG_SIZE, batch=BATCH,
        lr0=LR0, lrf=LRF, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY,
        warmup_epochs=WARMUP_EPOCHS, device=DEVICE, workers=NUM_WORKERS,
        patience=PATIENCE, project=RUNS_DIR, name='final',
        exist_ok=True, verbose=True, single_cls=True,
        fliplr=0.5, mosaic=1.0, scale=0.5,
    )
    del model; torch.cuda.empty_cache(); gc.collect()
    print('Training done.')

In [ ]:
# === Luu backbone va danh gia ===
best_pt = os.path.join(RUNS_DIR, 'final', 'weights', 'best.pt')
if os.path.exists(best_pt):
    shutil.copy2(best_pt, BACKBONE_PATH)
    print(f'Backbone saved: {BACKBONE_PATH}')
else:
    print('ERROR: best.pt not found, check training.')

# Evaluation
if os.path.exists(BACKBONE_PATH):
    model = YOLO(BACKBONE_PATH)
    metrics = model.val(data=YAML_PATH, imgsz=IMG_SIZE, batch=BATCH,
                        workers=NUM_WORKERS, verbose=False)
    m = metrics.results_dict
    print(f'mAP@0.5:      {m.get("metrics/mAP50(B)", 0):.4f}')
    print(f'mAP@0.5:0.95: {m.get("metrics/mAP50-95(B)", 0):.4f}')
    print(f'Precision:    {m.get("metrics/precision(B)", 0):.4f}')
    print(f'Recall:       {m.get("metrics/recall(B)", 0):.4f}')
    del model; torch.cuda.empty_cache(); gc.collect()

# Loss curve
csv_path = os.path.join(RUNS_DIR, 'final', 'results.csv')
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]
    fig, ax = plt.subplots(figsize=(10, 4))
    if 'train/box_loss' in df.columns:
        train_loss = (df['train/box_loss']
                      + df.get('train/cls_loss', 0)
                      + df.get('train/dfl_loss', 0))
        val_loss   = (df['val/box_loss']
                      + df.get('val/cls_loss', 0)
                      + df.get('val/dfl_loss', 0))
        ep = range(1, len(train_loss) + 1)
        ax.plot(ep, train_loss, label='Train Loss')
        ax.plot(ep, val_loss, '--', label='Val Loss')
    ax.set(title='SeaDroneSee RGB Backbone -- Loss', xlabel='Epoch', ylabel='Loss')
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    save_path = os.path.join(RUNS_DIR, 'loss_curves.png')
    plt.savefig(save_path, dpi=150); plt.show()
    print(f'Loss curves: {save_path}')